In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

In [ ]:
ruta_df1 = r"C:\Users\danie\OneDrive\Escritorio\DATA\dataset_final_v4p.csv"
ruta_df2 = r"C:\Users\danie\OneDrive\Escritorio\DATA\definitivo_v4p.csv"

df1 = pd.read_csv(ruta_df1)
df2 = pd.read_csv(ruta_df2)

print("df1:", df1.shape)
print("df2:", df2.shape)

In [ ]:
# Inspección inicial
print("DF1:")
display(df1.head())
print(df1.shape)

# Ver nombres de columnas
print("\nColumnas DF1:")
print(df1.columns)

In [ ]:
print("DF2 (clean):")
display(df2.head())
print(df2.shape)

# Ver nombres de columnas
print("\nColumnas DF2:")
print(df2.columns)

In [ ]:
df1.info() #Información general
display(df1.describe()) #Estadísticas descriptivas: cuenta, media, desviación, minimo, maximo y percentiles.
df1.isnull().sum() # Conteo de valores que no están por columna.

In [ ]:
df2.info()
display(df2.describe()) 
df2.isnull().sum()

In [ ]:
# Cada fila representa una estancia en UCI
n_pacientes = df1['subject_id'].nunique()
n_admisiones = df1['hadm_id'].nunique()
n_estancias = df1['stay_id'].nunique()

print(f"Pacientes únicos (subject_id): {n_pacientes}")
print(f"Admisiones únicas (hadm_id):   {n_admisiones}")
print(f"Estancias únicas (stay_id):    {n_estancias}")
print(f"Ratio estancias/paciente:      {len(df1) / n_pacientes}")

In [ ]:
n_pacientes = df2['subject_id'].nunique()
n_admisiones = df2['hadm_id'].nunique()
n_estancias = df2['stay_id'].nunique()

print(f"Pacientes únicos (subject_id): {n_pacientes}")
print(f"Admisiones únicas (hadm_id):   {n_admisiones}")
print(f"Estancias únicas (stay_id):    {n_estancias}")
print(f"Ratio estancias/paciente:      {len(df2) / n_pacientes}")

In [ ]:
# describe() sobre variables numéricas. Excluimos identificadores y la etiqueta
# para mostrar solo lo que tiene sentido clínico.
cols_descriptivo = [c for c in df1.columns
                    if c not in ['subject_id', 'hadm_id', 'stay_id',
                                 'etiqueta_norad_3_12', 'horas_hasta_norad']]

df1[cols_descriptivo].describe().T

In [ ]:
# describe() sobre variables numéricas. 
# solo lo que tiene sentido clínico.
cols_descriptivo = [c for c in df2.columns
                    if c not in ['subject_id', 'hadm_id', 'stay_id',
                                 'etiqueta_norad_3_12', 'horas_hasta_norad']]

df2[cols_descriptivo].describe().T

In [ ]:
# describe() estratificado por la clase objetivo
variables_clave = ['anchor_age', 'lactato_max', 'creatinina_max', 'map_min',
                   'hr_max', 'gcs_min', 'ph_min', 'diuresis_ml_kg_3h']

resumen_por_clase = df1.groupby('etiqueta_norad_3_12')[variables_clave].agg(['mean', 'median'])
resumen_por_clase.T

In [ ]:
# describe() estratificado por la clase objetivo
variables_clave = ['anchor_age', 'lactato_max', 'creatinina_max', 'map_min',
                   'hr_max', 'gcs_min', 'ph_min', 'diuresis_ml_kg_3h']

resumen_por_clase = df2.groupby('etiqueta_norad_3_12')[variables_clave].agg(['mean', 'median'])
resumen_por_clase.T

In [ ]:
#VARIABLES CON VARIANZA MUY BAJA

excluir = ['subject_id', 'hadm_id', 'stay_id',
           'horas_hasta_norad', 'etiqueta_norad_3_12']
num_cols = [c for c in df2.select_dtypes(include=[np.number]).columns
            if c not in excluir]

# Calculamos el coeficiente de variación (std/|mean|) para comparar entre escalas
resumen_var = pd.DataFrame({
    'std': df2[num_cols].std(),
    'mean': df2[num_cols].mean(),
})
resumen_var['cv'] = resumen_var['std'] / resumen_var['mean']
resumen_var.sort_values('cv').head(20)

### Análisis de la variable objetivo

In [ ]:
# Conteo y porcentaje por clase
conteo = df2['etiqueta_norad_3_12'].value_counts()
prevalencia = df2['etiqueta_norad_3_12'].value_counts(normalize=True)

resumen_clases = pd.DataFrame({
    'n': conteo,
    'porcentaje': (100*prevalencia).round(2)
})
resumen_clases.index = ['Negativos (no norad en 3-12h)', 'Positivos (norad en 3-12h)']
print(resumen_clases)

In [ ]:
#¿Cuándo se inicia la noradrenalina dentro de la ventana 3-12h?
positivos = df2[df2['etiqueta_norad_3_12'] == 1]['horas_hasta_norad']

print(f"Estadísticos de horas_hasta_norad en positivos (n={len(positivos)}):")
print(positivos.describe().round(2).to_string())

In [ ]:
# Histograma de horas_hasta_norad en positivos
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(positivos, bins=np.arange(3, 13, 1), color='#ff7f0e', alpha=0.8)
ax.axvline(positivos.median(), color='navy', linestyle='--',
           linewidth=2, label=f'Mediana = {positivos.median():.1f} h')
ax.axvline(positivos.mean(), color='darkred', linestyle=':',
           linewidth=2, label=f'Media = {positivos.mean():.1f} h')
ax.set_xlabel('Horas desde el ingreso en UCI hasta el inicio de noradrenalina')
ax.set_ylabel('Nº de estancias')
ax.set_title(f'Distribución temporal del evento en la clase positiva (n={len(positivos)})')
ax.set_xticks(np.arange(3, 13, 1))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ¿Cómo se componen los negativos?
neg = df2[df2['etiqueta_norad_3_12'] == 0]

neg_norad_tardia = neg[neg['horas_hasta_norad'].notna()]
neg_sin_norad   = neg[neg['horas_hasta_norad'].isna()]

print(f"Total de negativos: {len(neg)}")
print(f"  - Recibieron noradrenalina tras las 12 h ({neg_norad_tardia['horas_hasta_norad'].min()}h a "
      f"{neg_norad_tardia['horas_hasta_norad'].max()}h): {len(neg_norad_tardia)}")
print(f"  - No recibieron noradrenalina durante la estancia:              {len(neg_sin_norad)}")

In [ ]:
# Para los negativos que sí recibieron norad (pero tardíamente), ver cuándo la iniciaron
fig, ax = plt.subplots(figsize=(10, 5))
datos = neg_norad_tardia['horas_hasta_norad']
# Limitar el eje x por legibilidad (hay alguna estancia muy larga)
datos_mostrar = datos[datos <= 200]
ax.hist(datos_mostrar, bins=40, color='#1f77b4', alpha=0.7)
ax.axvline(12, color='red', linestyle='--', linewidth=2,
           label='Corte 12 h (límite de la ventana)')
ax.set_xlabel('Horas desde el ingreso en UCI hasta el inicio de noradrenalina')
ax.set_ylabel('Nº de estancias')
ax.set_title(f'Negativos con noradrenalina tardía')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Identificar pacientes que aparecen con etiqueta positiva en una estancia
por_paciente = df2.groupby('subject_id')['etiqueta_norad_3_12'].agg(['sum', 'count'])  #nos quedamos con los unos y ceros de cada "montón" y nos quedamos con la suma y counteo
pacientes_mixtos = por_paciente[(por_paciente['sum'] > 0) &
                                (por_paciente['sum'] < por_paciente['count'])] #al menios una estancia positiva y una negativa (suma tiene que dar uno y count 2 o más)

print(f"Pacientes con etiqueta mixta entre sus estancias: {len(pacientes_mixtos)}")
print(f"  (positivo en al menos una estancia, negativo en al menos otra)\n")

if len(pacientes_mixtos) > 0:
    print("Desglose:")
    print(pacientes_mixtos.head(10).to_string())